In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient

In [0]:
dbutils.widgets.text('source_schema', 'kp_catalog.mimic_incr')
source_schema = dbutils.widgets.get('source_schema')

dbutils.widgets.text('target_schema', 'kp_catalog.hls_ml')
target_schema = dbutils.widgets.get('target_schema')

In [0]:
client = VectorSearchClient()

In [0]:
endpoint_name = 'one-env-shared-endpoint-3'
try:
  client.get_endpoint(endpoint_name)
except Exception:
  client.create_endpoint(
    name=endpoint_name,
    endpoint_type="STANDARD" # or "STORAGE_OPTIMIZED"
  )

In [0]:
client.create_endpoint(
    name="one-env-shared-endpoint-3",
    endpoint_type="STANDARD" # or "STORAGE_OPTIMIZED"
)

In [0]:

index_name = f"{target_schema}.discharge_index"
i = client.get_index(endpoint_name,index_name)

In [0]:
i.

In [0]:
index = client.create_delta_sync_index_and_wait(
  endpoint_name=endpoint_name,
  source_table_name=f"{source_schema}.discharge",
  index_name=f"{target_schema}.discharge_index",
  pipeline_type="TRIGGERED",
  primary_key="note_id",
  embedding_source_column="text",
  embedding_model_endpoint_name="databricks-bge-large-en",
  usage_policy_id='test-fe'
)

In [0]:
spark.sql(f"select count(1) from {source_schema}.discharge").display()

In [0]:
spark.sql(f"select count(1) from {source_schema}.discharge where charttime > '2025-01-01'").display()

In [0]:
spark.sql(f"select date_trunc('month',charttime),count(1) from {source_schema}.discharge group by 1").display()

Databricks visualization. Run in Databricks to view.

In [0]:
dbutils.fs.ls('/Volumes/kp_catalog/hls_ml/icd10')

In [0]:
%sh
unzip "/Volumes/kp_catalog/hls_ml/icd10/icd10cm-Code Descriptions-2026.zip" -d /Volumes/kp_catalog/hls_ml/icd10/descriptions

In [0]:
import pandas as pd

column_names = ['order_number', 'icd10', 'header', 'short_description','long_description']  # Replace with your actual column names
df = pd.read_fwf(
    '/Volumes/kp_catalog/hls_ml/icd10/descriptions/icd10cm-order-2026.txt',
    names=column_names,
    dtype=str
)
display(df)

In [0]:
df.head()

In [0]:
spark.read.csv('/Volumes/kp_catalog/hls_ml/icd10/descriptions/icd10cm-order-2026.txt')